In [ ]:
#bash code
#normalA
nanopolish index -d /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy/workspace /root/sunxh/WaveCrossMamba/contrast/IVT_normalA.fastq

minimap2 -ax map-ont -t 8 /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa \
/root/sunxh/WaveCrossMamba/contrast/IVT_normalA.fastq | samtools sort -o /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.sorted.bam \
-T /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.sorted.tmp

samtools index /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.sorted.bam

nanopolish eventalign \
    -t 35 --signal-index\
    --reads /root/sunxh/WaveCrossMamba/contrast/IVT_normalA.fastq \
    --bam /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.sorted.bam \
    --genome /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa \
    --scale-events > /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.eventalign.txt

m6anet dataprep --eventalign /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy.eventalign.txt \
                --out_dir /root/sunxh/WaveCrossMamba/contrast/ --n_processes 16

m6anet inference --input_dir /root/sunxh/WaveCrossMamba/contrast/ \
--out_dir /root/sunxh/WaveCrossMamba/contrast/ELIGOS_m6anet/normalA/  --n_processes 4 --num_iterations 1000

nanopolish index -d /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy/workspace /root/sunxh/WaveCrossMamba/contrast/IVT_m6A.fastq

minimap2 -ax map-ont -t 8 /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa \
/root/sunxh/WaveCrossMamba/contrast/IVT_m6A.fastq | samtools sort -o /root/sunxh/WaveCrossMamba/contrast/K562.sorted.bam \
-T /root/sunxh/WaveCrossMamba/contrast/K562.sorted.tmp

samtools index /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy.sorted.bam

nanopolish eventalign \
    -t 35 --signal-index\
    --reads /root/sunxh/WaveCrossMamba/contrast/IVT_m6A.fastq \
    --bam /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy.sorted.bam \
    --genome /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa \
    --scale-events > /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy.eventalign.txt

m6anet dataprep --eventalign /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy.eventalign.txt \
                --out_dir /home/wuyou/Projects/paper/ELIGOS_m6A/nanopolish --n_processes 16

m6anet inference --input_dir /home/wuyou/Projects/paper/ELIGOS_m6A/nanopolish \
--out_dir /root/sunxh/WaveCrossMamba/contrast/ELIGOS_m6anet/m6A  --n_processes 4 --num_iterations 1000

In [1]:
#calculate ROC and PR
import pandas as pd
fprs,tprs,precisions,recalls,normalization_roc,normalization_pr=[],[],[],[],[],[]
from sklearn.metrics import roc_curve,auc,roc_auc_score,precision_recall_curve
batch_y,probabilities=[],[]
with open("/root/sunxh/WaveCrossMamba/contrast/ELIGOS_m6anet/normalA/data.indiv_proba.csv") as f:
    count=0
    for i,line in enumerate(f):
        if "probability" in line:
            continue
        if i%1==0:
            line=line.rstrip()
            probability=float(line.split(",")[-1])
            batch_y.append(0)
            probabilities.append(probability)
            if count>500:
                break
            
with open("/root/sunxh/WaveCrossMamba/contrast/ELIGOS_m6anet/m6A/data.indiv_proba.csv") as f:
    count=0
    for i,line in enumerate(f):
        if "probability" in line:
            continue
        if i%1==0:
            line=line.rstrip()
            probability=float(line.split(",")[-1])
            batch_y.append(1)
            probabilities.append(probability)
            if count>500:
                break

fpr,tpr,thersholds=roc_curve(batch_y,probabilities)
precision,recall,thersholds=precision_recall_curve(batch_y,probabilities)
roc_auc=auc(fpr,tpr)
pr_auc=auc(recall,precision)


fprs.extend(fpr)
tprs.extend(tpr)
precisions.extend(precision)
recalls.extend(recall)
normalization_roc.extend(["m6anet AUC %.2f" %roc_auc]*len(fpr))
normalization_pr.extend(["m6anet AUC %.2f" %pr_auc]*len(precision))
# 转换为 Pandas DataFrame
df_roc = pd.DataFrame({"FPR": fprs, "TPR": tprs, "AUC": normalization_roc})
df_pr = pd.DataFrame({"Precision": precisions, "Recall": recalls, "AUC": normalization_pr})

# 保存为 CSV
df_roc.to_csv("m6anet_roc.csv", sep=",", index=False)  
df_pr.to_csv("m6anet_pr.csv", sep=",", index=False)    
print(roc_auc)
print(pr_auc)

0.6769769688110326
0.7354046067340803


In [ ]:
#bash code
#normalA

nanopolish index -d /root/sunxh/WaveCrossMamba/human_cell/K562_guppy/workspace /root/sunxh/WaveCrossMamba/human_cell/K562.fastq

minimap2 -ax map-ont -t 8 /root/sunxh/data/fna/GRCh38_latest_rna.fna \
/root/sunxh/WaveCrossMamba/human_cell/K562.fastq | samtools sort -o /root/sunxh/WaveCrossMamba/human_cell/K562.sorted.bam \
-T /root/sunxh/WaveCrossMamba/human_cell/K562.sorted.tmp

samtools index /root/sunxh/WaveCrossMamba/human_cell/K562.sorted.bam 

nanopolish eventalign \
    -t 20 --signal-index\
    --reads /root/sunxh/WaveCrossMamba/human_cell/K562.fastq \
    --bam /root/sunxh/WaveCrossMamba/human_cell/K562.sorted.bam \
    --genome /root/sunxh/data/fna/GRCh38_latest_rna.fna \
    --scale-events > /root/sunxh/WaveCrossMamba/human_cell/K562.eventalign.txt

m6anet dataprep --eventalign /root/sunxh/WaveCrossMamba/human_cell/K562.eventalign.txt \
                --out_dir /root/sunxh/WaveCrossMamba/human_cell/m6anet --n_processes 8

m6anet inference --input_dir /root/sunxh/WaveCrossMamba/contrast/ \
--out_dir /root/sunxh/WaveCrossMamba/contrast/ELIGOS_m6anet/normalA/  --n_processes 4 --num_iterations 1000
